# Deploy the predictor and score Sally’s transactions

[Guide](README.md) · [00: prepare data](00_prepare_data.ipynb) · [01: experiment](01_experiment.ipynb) · [02: results](02_results.ipynb) · [03: predictor](03_predictor.ipynb)

Complete [02 — Results](02_results.ipynb). This notebook serves the selected predictor with the cluster's installed **kserve-autogluon-server** runtime. Use one of the two deployment methods below. REST scoring uses the original, unscaled, seven named features; it needs neither an ONNX export nor a scaler.

In [1]:
from pathlib import Path
import sys

# Works when Jupyter starts in either the repo root or automl/.
automl_dir = Path.cwd() if Path('workflow.py').exists() else Path.cwd() / 'automl'
sys.path.insert(0, str(automl_dir.resolve()))
import pandas as pd
from IPython.display import display, Markdown, FileLink
import workflow as wf

settings = wf.settings()  # Never display this mapping: it may contain credentials.

In [2]:
run = wf.read('run.json')
results = wf.read('results.json')
assert results['run_id'] == run['run_id'], 'Collect results for the current run first'
print('Selected model:', results['selected_model'])
print('Model storage URI:', results['predictor_uri'])
print('Connection bucket:', run['artifact_bucket'])
print('Storage path (within the bucket):', results['predictor_prefix'].rstrip('/'))

Selected model: RandomForestEntr_BAG_L1_FULL
Model storage URI: s3://YOUR_DATA_BUCKET/autogluon-tabular-training-pipeline/7c7e79be-552d-4959-9827-88e564970698/autogluon-models-training-3/a1b488a4-7a4d-42f8-a2fb-3f1c807b49bf/models_artifact/RandomForestEntr_BAG_L1_FULL/predictor/
Connection bucket: YOUR_DATA_BUCKET
Storage path (within the bucket): autogluon-tabular-training-pipeline/7c7e79be-552d-4959-9827-88e564970698/autogluon-models-training-3/a1b488a4-7a4d-42f8-a2fb-3f1c807b49bf/models_artifact/RandomForestEntr_BAG_L1_FULL/predictor


## Option A: deploy through the RHOAI dashboard

1. Open your project → Models and deploy a predictive model named `fraud-automl` (or the value of `MODEL_NAME`).
2. Select the installed AutoGluon runtime (`kserve-autogluon-server`) compatible with the managed training components. Choose an available CPU hardware profile, starting with 2 CPUs and 8 GiB memory, and one replica.
3. Select the S3 Connection for the **artifact bucket** shown above. Enter the printed storage path, ending in `predictor`, without a bucket name or `s3://`. Use the whole predictor directory.
4. Expose the endpoint, keep authentication enabled, configure access for your inference identity and wait for Ready.
5. Copy the base inference endpoint into `PREDICT_ENDPOINT` in `.env`. Use `PREDICT_AUTH_MODE=auto` and leave `PREDICT_TOKEN` empty to read current `oc login`/kubeconfig or workbench credentials on each request. The identity must have model inference access. A nonempty `PREDICT_TOKEN` explicitly overrides automatic credentials. Restart this kernel, then rerun the setup and results cells.
6. Skip Option B and continue at **Score Sally's transactions**.

If the data and artifact buckets differ, use a separate Connection pointing at the artifact bucket. The deployment requires only read access to the model files.

## Option B: create resources from this notebook

Skip these cells if you used the dashboard. Kubernetes access comes from the workbench ServiceAccount or local kubeconfig, independently of `RHOAI_TOKEN`. Your identity must be able to read the installed runtime template and Connection Secret and read/create model resources, ServiceAccounts, Roles and RoleBindings. The generated storage ServiceAccount receives only `get` access to this Connection Secret. These cells use the existing Connection and require its bucket to match the artifact bucket. The RHOAI model controller resolves the Connection into storage credentials.

First generate and inspect the concrete manifests. They contain no access keys. Runtime image and arguments come from the installed template; no upstream image is substituted.

In [ ]:
# Optional: run only for programmatic deployment.
api = wf.kubernetes_api()
rendered = wf.deployment_manifests(api)
import json
print(json.dumps(rendered, indent=2))

Run the following cell to create the reviewed resources. Rerunning it reuses matching resources, creates missing ones and prints `created`/`reused`. This also resumes partial creation. If an existing resource has different requested fields, the helper lists those fields and stops without overwriting it. Review the resource in the dashboard or choose new names and regenerate manifests. Configure model access for the identity used by auto authentication (or your explicit token) in the dashboard before scoring.

In [4]:
# Optional: creates storage SA/RBAC, ServingRuntime and InferenceService in this project.
wf.create_deployment(api, rendered)

ServiceAccount YOUR_PROJECT/fraud-automl-storage: reused
Role YOUR_PROJECT/fraud-automl-storage: reused
RoleBinding YOUR_PROJECT/fraud-automl-storage: reused
ServingRuntime YOUR_PROJECT/fraud-automl-runtime: created
InferenceService YOUR_PROJECT/fraud-automl: created


[{'kind': 'ServiceAccount',
  'name': 'fraud-automl-storage',
  'action': 'reused'},
 {'kind': 'Role', 'name': 'fraud-automl-storage', 'action': 'reused'},
 {'kind': 'RoleBinding', 'name': 'fraud-automl-storage', 'action': 'reused'},
 {'kind': 'ServingRuntime',
  'name': 'fraud-automl-runtime',
  'action': 'created'},
 {'kind': 'InferenceService', 'name': 'fraud-automl', 'action': 'created'}]

In [5]:
# Optional: wait for Ready and discover the inference base URL.
endpoint = wf.serving_endpoint(api)
print('Inference endpoint:', endpoint)

[('IngressReady', 'True', ''), ('PredictorReady', 'False', 'Deployment does not have minimum availability.'), ('Ready', 'False', 'Deployment does not have minimum availability.'), ('Stopped', 'False', '')]
[('IngressReady', 'True', ''), ('PredictorReady', 'True', 'ReplicaSet "fraud-automl-predictor-695b566d57" has successfully progressed.'), ('Ready', 'True', ''), ('Stopped', 'False', '')]
Inference endpoint: https://fraud-automl-YOUR_PROJECT.apps.YOUR_CLUSTER


## Score Sally's transactions

Use `PREDICT_ENDPOINT` after dashboard deployment, or the endpoint discovered above. By default, `PREDICT_AUTH_MODE=auto` reads the current kubeconfig or workbench bearer token for each request; leave `PREDICT_TOKEN` empty to use it. Renew `oc login` if a static login token expires. The identity still needs model inference access. For an intentionally public endpoint set `PREDICT_AUTH_MODE=none`; this omits the Authorization header without changing deployment authentication. All seven features are supplied. Keep scalar values in the DataFrame below; `wf.predict()` converts each feature into a one-element column list required by the AutoGluon KServe v1 runtime. The first example represents Sally buying coffee locally; the second represents an online purchase far from home and the previous transaction. Values for the two additional features are illustrative assumptions for this seven-feature example.

This is a serving smoke test, not a guarantee about the model's decisions. Inspect the returned predictions. The v1 response can contain predicted labels; do not interpret a label as a probability or apply the old ONNX threshold. For probabilities use the official predictor notebook's `predict_proba` cells.

In [6]:
settings = wf.settings()  # Read current configuration before scoring.
endpoint = settings.get('PREDICT_ENDPOINT', '').strip() or globals().get('endpoint')
if not endpoint:
    raise ValueError('Set PREDICT_ENDPOINT in .env and restart the kernel, or complete Option B')
sally = pd.DataFrame([
    {'distance_from_home': 0.5, 'distance_from_last_transaction': 0.0,
     'ratio_to_median_purchase_price': 1.0, 'repeat_retailer': 1.0,
     'used_chip': 1.0, 'used_pin_number': 1.0, 'online_order': 0.0},
    {'distance_from_home': 150.0, 'distance_from_last_transaction': 100.0,
     'ratio_to_median_purchase_price': 1.2, 'repeat_retailer': 0.0,
     'used_chip': 0.0, 'used_pin_number': 0.0, 'online_order': 1.0},
], index=['Sally buys coffee', 'Suspicious online purchase'])
display(sally)
response = wf.predict(endpoint, sally.to_dict(orient='records'))
display(pd.DataFrame({'scenario': sally.index, 'prediction': response['predictions']}))
print('Serving predict worked: True')

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order
Sally buys coffee,0.5,0.0,1.0,1.0,1.0,1.0,0.0
Suspicious online purchase,150.0,100.0,1.2,0.0,0.0,0.0,1.0


,scenario,prediction
0,Sally buys coffee,0
1,Suspicious online purchase,1


Serving predict worked: True


## Save evidence for the ticket

The report contains measured results; fill in branch/PR links and observed failure/OOM evidence before copying. It is not posted automatically. Stop the serving model and cancel unused runs through the dashboard when finished; remove only your own S3 experiment prefixes.

In [ ]:
print(wf.ticket_report())
display(FileLink(str((wf.OUTPUTS / 'ticket-comment.md').relative_to(Path.cwd()))))

Return to [the guide](README.md) or inspect your [experiment results](02_results.ipynb).